# Sparse-scaling benchmark: analysis

This notebook reads the tables `collect.py` writes and answers three questions:

1. Up to which `nawf` does each pipeline (dense, sparse from the bottom, sparse interior) run?
   What limits it: memory, time, or a refusal?
2. How do time, memory and sparsity scale with `nawf`? Where do the pipelines cross over?
3. What do the measurements say about the policy constants `dense_n_max` and `dense_ratio`, and
   from what size do the KPM/Sternheimer formulations of `handoff.md` become necessary?

Run `python collect.py` first (the finalize job does this automatically). The output root comes
from `config.py`, or from `BENCH_OUTPUT_ROOT` when that is set.

**Reading the accuracy panel.** As of 2026-10-06, `doubling_Hamiltonian` gives wrong band
velocities on doubled cells in both pipelines. It zeroes the cross-replica `Dnm`, and σ changes by
about 50% on an equivalent cell. Until that is fixed, the deviations of d > 0 cases measure this
bug and not the pipelines. The timing and memory panels are unaffected.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('.'))
from config import CONFIG

ROOT = os.environ.get('BENCH_OUTPUT_ROOT', CONFIG['output_root'])
df = pd.read_csv(os.path.join(ROOT, 'results.csv'))
sweep_path = os.path.join(ROOT, 'sweep.csv')
sw = pd.read_csv(sweep_path) if os.path.getsize(sweep_path) > 1 else pd.DataFrame()
GB = 1024.0**3

# palette: deep blue, vermillion, amber, then Okabe-Ito
PALETTE = ['#004488', '#D55E00', '#FFC20A', '#009E73', '#56B4E9', '#CC79A7', '#000000']
AMBER_TEXT = '#8A6800'
pipelines = list(dict.fromkeys(df['pipeline']))
COLOR = {p: PALETTE[i % len(PALETTE)] for i, p in enumerate(pipelines)}
MARKER = dict(zip(pipelines, 'osD^vP*'))
plt.rcParams.update({'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})

ok = df[df['status'] == 'ok']
failed = df[df['status'] != 'ok']
print('%d cases: %d ok, %d failed' % (len(df), len(ok), len(failed)))
df[['id', 'nawf', 'status', 'failure']].tail(len(df))

## Where each pipeline stops

Per pipeline: the largest `nawf` that ran, and the first size that failed, with the reason. The
reason is `oom`, `timeout`, or a Python error such as the dispatch refusal. Failures that the
scheduler caught come from `finalize.py` (sacct).

In [ ]:
rows = []
for p in pipelines:
    sub = df[df['pipeline'] == p].sort_values('nawf')
    good = sub[sub['status'] == 'ok']
    bad = sub[sub['status'] != 'ok']
    first_bad = bad.iloc[0] if len(bad) else None
    rows.append(
        {
            'pipeline': p,
            'largest ok nawf': int(good['nawf'].max()) if len(good) else None,
            'first failing nawf': int(first_bad['nawf']) if first_bad is not None else None,
            'reason': first_bad['failure'] if first_bad is not None else '(none failed)',
            'peak RSS/rank at largest ok (GB)': good['peak_rss_max'].iloc[-1] / GB
            if len(good)
            else None,
        }
    )
limits = pd.DataFrame(rows)
limits

## 1. Wall time against `nawf`

The *k-space* time covers every stage after the Hamiltonian is built: the interpolation, the
window probe, and the eigensolve plus `dos` plus `transport`. The sparse pipeline fuses these into
a single mesh pass, while dense runs them as separate stages, so the sum is what compares like for
like. Dashed lines are power-law fits over the successful sizes, and ✕ marks the first failure
of each pipeline.

The k-mesh shrinks with the cell (fixed physical k-density), so a flat line here means the cost
per cell is constant. Dense meshes are clamped to the H(R) grid at larger d (hollow markers), so
their total includes extra k-points; panel 4 has the per-k comparison.

In [ ]:
def powerlaw(x, y):
    """Exponent and prefactor of y = a x^p over the finite positive points."""
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if m.sum() < 2:
        return None
    p, loga = np.polyfit(np.log(x[m]), np.log(y[m]), 1)
    return p, np.exp(loga)


def mark_failures(ax, column, top):
    for p in pipelines:
        bad = failed[failed['pipeline'] == p].sort_values('nawf')
        if len(bad):
            ax.plot(bad['nawf'].iloc[0], top, 'x', color=COLOR[p], ms=11, mew=2.5, clip_on=False)


fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)
for ax, column, title in (
    (axes[0], 't_kspace', 'k-space work'),
    (axes[1], 't_setup', 'H(R) build + doubling'),
):
    top = ok[column].max() * 3 if len(ok) else 1
    for p in pipelines:
        sub = ok[ok['pipeline'] == p].sort_values('nawf')
        if not len(sub):
            continue
        clamped = sub['mesh_clamped'].fillna(False).astype(bool)
        ax.plot(sub['nawf'], sub[column], '-', color=COLOR[p], alpha=0.5)
        ax.plot(sub['nawf'][~clamped], sub[column][~clamped], MARKER[p], color=COLOR[p], label=p)
        ax.plot(sub['nawf'][clamped], sub[column][clamped], MARKER[p], mfc='none', color=COLOR[p])
        fit = powerlaw(sub['nawf'].values, sub[column].values)
        if fit:
            xs = np.geomspace(sub['nawf'].min(), sub['nawf'].max(), 50)
            ax.plot(xs, fit[1] * xs ** fit[0], '--', color=COLOR[p], lw=1)
            ax.annotate(
                'n$^{%.2f}$' % fit[0],
                (xs[-1], fit[1] * xs[-1] ** fit[0]),
                color=COLOR[p] if COLOR[p] != '#FFC20A' else AMBER_TEXT,
                fontsize=9,
                xytext=(4, 0),
                textcoords='offset points',
            )
    mark_failures(ax, column, top)
    ax.set_xscale('log', base=2)
    ax.set_yscale('log')
    ax.set_xlabel('nawf')
    ax.set_ylabel('seconds')
    ax.set_title(title)
axes[0].legend(fontsize=9)
fig.tight_layout()

## 2. Memory against `nawf`

The left panel shows the peak resident memory per MPI rank, which is the maximum over stages from
`VmHWM` (the memory that kills a job). The right panel shows the Hamiltonian alone: the dense
H(R) array against the sparse assembly plan, and one dense per-k H(k) for scale. The dotted line
is the node's memory divided by the ranks per node.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)
ax = axes[0]
for p in pipelines:
    sub = ok[ok['pipeline'] == p].sort_values('nawf')
    ax.plot(sub['nawf'], sub['peak_rss_max'] / GB, MARKER[p] + '-', color=COLOR[p], label=p)
node = df['node_mem_total'].dropna()
if len(node):
    ax.axhline(node.max() / GB, ls=':', color='k', lw=1)
    ax.annotate('node memory', (ok['nawf'].min(), node.max() / GB), fontsize=9, va='bottom')
mark_failures(ax, 'peak_rss_max', ok['peak_rss_max'].max() / GB * 3)
ax.set_ylabel('peak RSS per rank (GB)')
ax.set_title('process memory')
ax = axes[1]
dense = ok[ok['pipeline'] == 'dense'].sort_values('nawf')
if len(dense):
    ax.plot(
        dense['nawf'], dense['HRs_bytes'] / GB, 'o-', color=COLOR['dense'], label='dense H(R) array'
    )
for p in pipelines:
    sub = ok[(ok['pipeline'] == p) & ok['plan_bytes'].notna()].sort_values('nawf')
    if len(sub):
        ax.plot(
            sub['nawf'],
            sub['plan_bytes'] / GB,
            MARKER[p] + '-',
            color=COLOR[p],
            label=p + ': bond-list plan',
        )
allx = df['nawf'].dropna().sort_values().unique()
ax.plot(allx, 16.0 * allx**2 / GB, ':', color='#000000', label='one dense H(k)')
ax.set_title('Hamiltonian storage')
ax.set_ylabel('GB')
for a in axes:
    a.set_xscale('log', base=2)
    a.set_yscale('log')
    a.set_xlabel('nawf')
axes[1].legend(fontsize=8)
fig.tight_layout()

## 3. Sparsity of H(k) against `nawf`

For a fixed real-space cutoff, the nonzeros per row of H(k) saturate once the cell is larger
than the cutoff sphere, and the fill fraction then falls as 1/nawf. The sparse algorithms only
pay off on the right of that knee. The Si PAO hoppings decay slowly, which puts the knee at
nawf ~ 10⁴ for `bond_order=36` and around 10³ for `bond_order=8`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharex=True)
for p in pipelines:
    sub = ok[(ok['pipeline'] == p) & ok['nnz_hk_per_row'].notna()].sort_values('nawf')
    if not len(sub):
        continue
    lab = '%s (bond_order %s)' % (p, sub['bond_order'].iloc[0])
    axes[0].plot(sub['nawf'], sub['nnz_hk_per_row'], MARKER[p] + '-', color=COLOR[p], label=lab)
    axes[1].plot(sub['nawf'], sub['density_hk'], MARKER[p] + '-', color=COLOR[p], label=lab)
axes[0].plot(allx, allx, ':', color='k', lw=1, label='dense row (= nawf)')
axes[0].set_ylabel('nonzeros per row of H(k)')
axes[1].set_ylabel('fill fraction of H(k)')
for a in axes:
    a.set_xscale('log', base=2)
    a.set_yscale('log')
    a.set_xlabel('nawf')
axes[0].legend(fontsize=8)
fig.tight_layout()

## 4. Per-k solve cost, and the measured `dense_n_max`

The left panel shows the eigensolve time per k-point and per pool: the sparse mesh-pass module
time or the dense `pao_eigh`, times the k-points one pool carries, plotted against `nawf` and
coloured by pipeline. The right panel, from the solver sweep, shows the extra memory (traced
allocations) one solve needs per kernel, with the LAPACK fit `c·n²`. The constant `c` turns a
memory budget per rank into a measured `dense_n_max = sqrt(budget / c)`. The current default of
4096 is drawn for comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
ax = axes[0]
for p in pipelines:
    sub = ok[(ok['pipeline'] == p) & ok['t_per_k'].notna()].sort_values('nawf')
    if not len(sub):
        continue
    ax.plot(sub['nawf'], sub['t_per_k'], MARKER[p] + '-', color=COLOR[p], label=p)
    fit = powerlaw(sub['nawf'].values, sub['t_per_k'].values)
    if fit:
        ax.annotate(
            'n$^{%.2f}$' % fit[0],
            (sub['nawf'].iloc[-1], sub['t_per_k'].iloc[-1]),
            fontsize=9,
            xytext=(4, 0),
            textcoords='offset points',
            color=COLOR[p] if COLOR[p] != '#FFC20A' else AMBER_TEXT,
        )
ax.axvline(4096, ls=':', color='k', lw=1)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('nawf')
ax.set_ylabel('seconds per k-point per pool')
ax.legend(fontsize=8)
ax.set_title('per k: sparse mesh pass / dense pao_eigh')

ax = axes[1]
c_fit = None
if len(sw):
    for kernel, color in (('dense', PALETTE[0]), ('sparse', PALETTE[1])):
        sub = sw[(sw['kernel'] == kernel) & sw['scratch_bytes'].notna()]
        if not len(sub):
            continue
        # one point per (n, kernel): the scratch at the largest measured nev
        last = sub.sort_values('nev').groupby('n').tail(1).sort_values('n')
        ax.plot(
            last['n'],
            last['scratch_bytes'] / GB,
            'o-' if kernel == 'dense' else 's-',
            color=color,
            label={'dense': 'LAPACK (zheevr)', 'sparse': 'ARPACK, largest nev run'}[kernel],
        )
        if kernel == 'dense':
            c_fit = float(np.median(last['scratch_bytes'] / last['n'] ** 2))
    if c_fit:
        xs = np.geomspace(sw['n'].min(), 65536, 50)
        ax.plot(
            xs,
            c_fit * xs**2 / GB,
            '--',
            color=PALETTE[0],
            lw=1,
            label='c n$^2$, c = %.1f B' % c_fit,
        )
ax.axvline(4096, ls=':', color='k', lw=1)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.set_xlabel('n')
ax.set_ylabel('per-k scratch (GB)')
ax.legend(fontsize=8)
ax.set_title('per-k scratch memory')
fig.tight_layout()

if c_fit:
    ranks = df['mpi_ranks'].dropna().max()
    for budget_gb in (4, 8, 16, 32, 64):
        print(
            'budget %3d GB/rank -> measured dense_n_max ~ %6d  (c = %.1f B/n^2)'
            % (budget_gb, int(np.sqrt(budget_gb * GB / c_fit)), c_fit)
        )

### 4b. Solver crossover: the measured `dense_ratio`

This is the ARPACK time over the LAPACK time at fixed `n`, plotted against `nev/n`, with one
curve per size. ARPACK wins below 1. The `nev/n` where a curve crosses 1 is the measured
`dense_ratio(n)`, and the current default of 1/8 is drawn for comparison. ARPACK points stop once
it is `stop_ratio` times slower than LAPACK.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.6))
cross = []
if len(sw):
    piv = sw.pivot_table(
        index=['n', 'nev_fraction'], columns='kernel', values='seconds'
    ).reset_index()
    for i, (n, sub) in enumerate(piv.groupby('n')):
        if 'sparse' not in sub or 'dense' not in sub:
            continue
        sub = sub.dropna(subset=['sparse', 'dense'])
        if not len(sub):
            continue
        r = sub['sparse'] / sub['dense']
        ax.plot(sub['nev_fraction'], r, 'o-', color=PALETTE[i % len(PALETTE)], label='n = %d' % n)
        below = sub['nev_fraction'][r < 1]
        cross.append({'n': n, 'ARPACK faster up to nev/n': below.max() if len(below) else 0.0})
ax.axhline(1, color='k', lw=1)
ax.axvline(0.125, ls=':', color='k', lw=1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('nev / n')
ax.set_ylabel('t(ARPACK) / t(LAPACK)')
ax.legend(fontsize=8)
fig.tight_layout()
pd.DataFrame(cross)

## 5. Accuracy per cell

Each run's per-cell DoS and σ_xx is compared on the energy range it shares with the reference,
as a relative L2 deviation. Two references are used:

- **dense d=0**, the exact answer;
- **the same pipeline at d=0**, which isolates the effect of doubling from that of the bond
  cutoff or the window.

The doubling-velocity bug noted at the top dominates the d > 0 points of both references until it
is fixed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharex=True)
for ax, key, title in ((axes[0], 'dos', 'DoS per cell'), (axes[1], 'sigma', 'sigma_xx')):
    for p in pipelines:
        sub = ok[ok['pipeline'] == p].sort_values('nawf')
        ax.plot(
            sub['nawf'],
            sub['dev_%s_vs_dense0' % key],
            MARKER[p] + '-',
            color=COLOR[p],
            label=p + ' vs dense d=0',
        )
        ax.plot(
            sub['nawf'],
            sub['dev_%s_vs_self0' % key],
            MARKER[p] + ':',
            mfc='none',
            color=COLOR[p],
            label=p + ' vs own d=0',
        )
    ax.set_xscale('log', base=2)
    ax.set_yscale('symlog', linthresh=1e-6)
    ax.set_xlabel('nawf')
    ax.set_ylabel('relative L2 deviation')
    ax.set_title(title)
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()

## 6. Summary: where each formulation is needed

- **Crossover.** The smallest `nawf` at which a sparse pipeline's k-space time beats the dense
  pipeline's, at the same mesh.
- **Measured limits.** The largest successful `nawf` per pipeline (the table above) and the
  `dense_n_max` for a given memory per rank (panel 4).
- **When `handoff.md` is needed.** The from-the-bottom pipeline (`sparse`) and every full-spectrum
  property scale as the per-k LAPACK solve. Past its measured limit only `sparse_interior`
  survives, and only for properties near E_F. Fermi-sea and interband properties beyond that size
  are what the KPM and Sternheimer variants are for. The threshold is the measured
  `dense_n_max`, not the default of 4096.

In [ ]:
dense_t = ok[ok['pipeline'] == 'dense'].set_index('nawf')['t_kspace']
summary = []
for p in pipelines:
    if p == 'dense':
        continue
    sub = ok[ok['pipeline'] == p].set_index('nawf')['t_kspace']
    common = sorted(set(sub.index) & set(dense_t.index))
    faster = [n for n in common if sub[n] < dense_t[n]]
    good = ok[ok['pipeline'] == p]
    summary.append(
        {
            'pipeline': p,
            'faster than dense from nawf': min(faster) if faster else None,
            'largest ok nawf': int(good['nawf'].max()) if len(good) else None,
            'kernel at largest': good.sort_values('nawf')['kernel'].iloc[-1] if len(good) else None,
        }
    )
pd.DataFrame(summary)